# Cache with Random Replacement (RR) Policy

This example demonstrates:
- Open network with cache node
- RR (Random Replacement) strategy
- 5 items, cache capacity of 2
- Zipf access pattern (skewed distribution)
- Hit ratio analysis

In [ ]:
from line_solver import *
import numpy as np

In [ ]:
def cache_replc_rr():

    model = Network('model')

    n = 5  # number of items
    m = 2  # cache capacity

    source = Source(model, 'Source')
    cache_node = Cache(model, 'Cache', n, m, ReplacementStrategy.RR)
    sink = Sink(model, 'Sink')

    job_class = OpenClass(model, 'InitClass', 0)
    hit_class = OpenClass(model, 'HitClass', 0)
    miss_class = OpenClass(model, 'MissClass', 0)

    source.set_arrival(job_class, Exp(2))

    # Zipf-like item references
    p_access = Zipf(1.4, n)
    cache_node.set_read(job_class, p_access)

    cache_node.set_hit_class(job_class, hit_class)
    cache_node.set_miss_class(job_class, miss_class)

    P = model.init_routing_matrix()
    P.set(job_class, job_class, source, cache_node, 1.0)
    P.set(hit_class, hit_class, cache_node, sink, 1.0)
    P.set(miss_class, miss_class, cache_node, sink, 1.0)

    model.link(P)

    # Run multiple solvers
    return model, cache_node

In [ ]:
GlobalConstants.set_verbose(VerboseLevel.STD)
model, cache_node = cache_replc_rr()
solver = np.array([], dtype=object)
solver = np.append(solver, CTMC(model, keep=False, cutoff=1))

In [ ]:
model.reset()
solver = np.append(solver, SSA(model, samples=10000, verbose=True, method='serial', seed=23000))

In [ ]:
model.reset()
solver = np.append(solver, MVA(model))

In [ ]:
model.reset()
solver = np.append(solver, NC(model))

In [ ]:
model.reset()
solver = np.append(solver, FLD(model, method='rmf'))

In [ ]:
avg_node_table = np.empty(len(solver), dtype=object)
for s in range(len(solver)):
    print(f'\nSOLVER: {solver[s].get_name()}')
    avg_node_table[s] = solver[s].avg_node_table()
    print(avg_node_table[s])

In [ ]:
print(f'\nHit Ratio: {cache_node.get_hit_ratio()}')
print(f'Miss Ratio: {cache_node.get_miss_ratio()}')